# Unit 5 - Work with data using Spark SQL

In [1]:
# Notebook Setup
# Attach to Lakehouse
# Import necessary libraries if needed

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

StatementMeta(, ed1e4957-6dfd-45f9-b71d-e9d6eeeb78be, 3, Finished, Available, Finished, False)

In [22]:
# Load the data we used previously 'products.csv'(with header)

df = spark.read.load("Files/data/products.csv", format="csv", header=True, inferSchema=True)
display(df.limit(5))

StatementMeta(, ed1e4957-6dfd-45f9-b71d-e9d6eeeb78be, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d9f44d7a-c0d6-4d27-85e4-efcf56bfdaa1)

## 1. Creating Temporary View

In [23]:
# Register the DataFrame as a SQL view.
df.createOrReplaceTempView("products_view")

StatementMeta(, ed1e4957-6dfd-45f9-b71d-e9d6eeeb78be, 25, Finished, Available, Finished, False)

In [24]:
# Test with a simple query:
spark.sql("SELECT * FROM products_view LIMIT 5").show()

StatementMeta(, ed1e4957-6dfd-45f9-b71d-e9d6eeeb78be, 26, Finished, Available, Finished, False)

+---------+-------------+--------------+---------+
|ProductID|  ProductName|      Category|ListPrice|
+---------+-------------+--------------+---------+
|        1| Mountain-100|Mountain Bikes|  2499.99|
|        2|     Road-150|    Road Bikes|   3578.0|
|        3|HL Road Frame|    Road Bikes|   1431.5|
|        4| Mountain-200|Mountain Bikes|   2999.0|
|        5| Touring-1000| Touring Bikes|   2384.0|
+---------+-------------+--------------+---------+



## 2. Save as Delta Table

In [25]:
# Persist the DataFrame as a managed Delta table.
df.write.format("delta").mode("overwrite").saveAsTable("products")

StatementMeta(, ed1e4957-6dfd-45f9-b71d-e9d6eeeb78be, 27, Finished, Available, Finished, False)

In [26]:
# Verify with SQL:
spark.sql("SELECT COUNT(*) FROM products").show()

StatementMeta(, ed1e4957-6dfd-45f9-b71d-e9d6eeeb78be, 28, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|       6|
+--------+



## 3. Create External Table

##  4. Query with Spark SQL API

In [37]:
# Run SQL queries inside PySpark
bikes_df = spark.sql(
    "SELECT ProductID, ProductName, ListPrice \
     FROM products \
     WHERE Category IN ('Mountain Bikes','Road Bikes')"
)
display(bikes_df)

StatementMeta(, ed1e4957-6dfd-45f9-b71d-e9d6eeeb78be, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0af11c39-80f7-4ecf-8d08-dae2f05ff6bf)

## 5. Query with `%%sql` Magic

In [39]:
%%sql
SELECT Category, COUNT(ProductID) AS ProductCount
FROM products
GROUP BY Category
ORDER BY Category

StatementMeta(, ed1e4957-6dfd-45f9-b71d-e9d6eeeb78be, 42, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

## 6. Partitioned Delta Table

In [40]:
# Save partitioned data for performance
df.write.partitionBy("Category").format("delta").mode("overwrite").saveAsTable("partitioned_products")

StatementMeta(, ed1e4957-6dfd-45f9-b71d-e9d6eeeb78be, 43, Finished, Available, Finished, False)

In [41]:
# Query partitioned table
spark.sql("SELECT * FROM partitioned_products WHERE Category='Road Bikes'").show()

StatementMeta(, ed1e4957-6dfd-45f9-b71d-e9d6eeeb78be, 44, Finished, Available, Finished, False)

+---------+-------------+----------+---------+
|ProductID|  ProductName|  Category|ListPrice|
+---------+-------------+----------+---------+
|        2|     Road-150|Road Bikes|   3578.0|
|        3|HL Road Frame|Road Bikes|   1431.5|
+---------+-------------+----------+---------+

